# Демонстрация безлинзовой реконструкции

Данный демо-ноутбук клонирует репозиторий, устанавливает зависимости, скачивает чекпоинт LeADMM-20, получает тестовый датасет из ZIP-архива, выполняет инференс и рассчитывает метрики.

## 1. Клонирование репозитория

Скачиваем репозиторий.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/glebustim/dl-hw05-project-glebustim.git"
REPOSITORY_NAME = "dl-hw05-project-glebustim"
REPOSITORY_DIR = Path.cwd() / REPOSITORY_NAME
subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
os.chdir(REPOSITORY_DIR)

## 2. Установка зависимостей

Устанавливаются зависимости проекта из `requirements.txt`.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt",
    ],
    check=True,
)

## 3. Настройка скачивания датасета и инференса

Вставьте в `DATASET_ZIP_URL` ссылку общего доступа на ZIP-архив Google Drive. Архив может содержать либо каталог с `test/lensless`, `test/masks` и опциональным `test/lensed`, либо сразу эти три каталога.

По умолчанию реконструкции сохраняются в `data/saved/saved_reconstructions/leadmm_20/test/`. Измените только `OUTPUT_NAME`, чтобы выбрать другой каталог.

Для выбора другой модели или чекпоинта также нужно поменять соответствующие переменные.

In [ ]:
DATASET_ZIP_URL = ""  # Вставьте сюда ссылку Google Drive на ZIP-архив.

MODEL_REPOSITORY = "glebustim/modular_leadmm_5_pre"
CHECKPOINT_FILENAME = "checkpoint-epoch8.pth"
MODEL_NAME = "modular_leadmm_5_pre"
OUTPUT_NAME = "saved_reconstructions/modular_leadmm_5_pre"

## 4. Скачивание и подготовка датасета

Эта ячейка скачивает архив, распаковывает его и помещает тестовую выборку в `DigiCam-Mirflickr-MultiMask-10/test/`.

In [ ]:
import shutil
import zipfile

import gdown

DATASET_ROOT = REPOSITORY_DIR / "DigiCam-Mirflickr-MultiMask-10K"
DOWNLOAD_DIR = REPOSITORY_DIR / "downloaded_dataset"
ARCHIVE_PATH = DOWNLOAD_DIR / "dataset.zip"
EXTRACT_DIR = DOWNLOAD_DIR / "extracted"

if DOWNLOAD_DIR.exists():
    shutil.rmtree(DOWNLOAD_DIR)
DOWNLOAD_DIR.mkdir()

gdown.download(DATASET_ZIP_URL, str(ARCHIVE_PATH), fuzzy=True)

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    archive.extractall(EXTRACT_DIR)

def find_test_split(root):
    for candidate in [root, *root.rglob("*")]:
        if not candidate.is_dir():
            continue
        if (candidate / "test" / "lensless").is_dir() and (candidate / "test" / "masks").is_dir():
            return candidate / "test"
        if (candidate / "lensless").is_dir() and (candidate / "masks").is_dir():
            return candidate

source_test_dir = find_test_split(EXTRACT_DIR)
if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)
shutil.copytree(source_test_dir, DATASET_ROOT / "test")

## 5. Скачивание чекпоинта

Чекпоинт LeADMM-20 скачивается из Hugging Face в `checkpoints/`.

In [ ]:
from huggingface_hub import hf_hub_download

checkpoint_path = hf_hub_download(
    repo_id=MODEL_REPOSITORY,
    filename=CHECKPOINT_FILENAME,
    local_dir=REPOSITORY_DIR / "checkpoints" / MODEL_NAME,
)

## 6. Инференс

Запускается `inference.py` на всех изображениях из скачанной тестовой выборки. `dataloader.drop_last=false` гарантирует, что будет обработано каждое изображение, включая неполный последний батч.

In [ ]:
inference_command = [
    sys.executable,
    "inference.py",
    f"model={MODEL_NAME}",
    "~datasets.train",
    "dataloader.shuffle=false",
    "dataloader.drop_last=false",
    f"datasets.test.data_dir={DATASET_ROOT}",
    "datasets.test.split=test",
    f"inferencer.from_pretrained={checkpoint_path}",
    f"inferencer.save_path={OUTPUT_NAME}",
]
subprocess.run(inference_command, check=True)

RECONSTRUCTION_DIR = REPOSITORY_DIR / "data" / "saved" / OUTPUT_NAME / "test"

## 7. Визуализация

Визуализируем посчитанные результаты.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

lensless_dir = DATASET_ROOT / "test" / "lensless"
lensed_dir = DATASET_ROOT / "test" / "lensed"
lensless_path = next(sorted(lensless_dir.glob("*.png")))
image_id = lensless_path.stem
reconstruction_path = RECONSTRUCTION_DIR / f"{image_id}.png"

panels = [("Беслинзовое", lensless_path), ("Реконструкция", reconstruction_path)]
lensed_path = lensed_dir / f"{image_id}.png"
if lensed_path.is_file():
    panels.append(("Исходное изображение", lensed_path))

figure, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
for axis, (title, path) in zip(axes, panels):
    with Image.open(path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(title)
    axis.axis("off")
figure.tight_layout()
plt.show()

## 8. Расчёт метрик

Если архив содержит исходные изображения в `lensed/`, запускается `calculate_metrics.py`. Скрипт печатает MSE, LPIPS, PSNR и SSIM для сохранённых реконструкций.

In [ ]:
if lensed_dir.is_dir() and any(lensed_dir.glob("*.png")):
    metrics_command = [
        sys.executable,
        "calculate_metrics.py",
        "--lensed_dir",
        str(lensed_dir),
        "--recon_dir",
        str(RECONSTRUCTION_DIR),
    ]
    subprocess.run(metrics_command, check=True)